# SPOTnet — Step 01: Specular Library Creation

This notebook extracts real specular reflection patches from endoscopic frames using HSV-based detection.  
The resulting patch library is used to train SPOTnet in the subsequent notebooks.

**Pipeline:**
```
Frames → HSV Detection → Morphological Refinement → Quality Filtering → Patch Library
```

**Output structure:**
```
real_specular_library_v1/
├── patches/        ← cropped RGB patches
├── masks/          ← binary specular masks
├── meta/           ← JSON metadata per patch
└── library_summary.json
```

## 1. Imports

In [ ]:
import os, json, math
import cv2
import numpy as np
from tqdm import tqdm

## 2. Configuration
Adjust paths and parameters below to match your setup.

In [ ]:
# ===================== CONFIG =====================
FRAMES_DIR = "frames"                          # folder containing your endoscopic frames
OUT_ROOT   = "real_specular_library_v1"        # output library root

# HSV Detection Parameters
HSV_V_THR = 200          # brightness threshold (0-255)
HSV_S_THR = 35           # saturation threshold (0-255)
MIN_CONTOUR_AREA = 3     # minimum contour area (pixels)
DILATION_KERNEL_SIZE = 7 # dilation kernel size

# Quality Filters
MIN_EXEMPLAR_AREA = 25           # ignore tiny speckles
MAX_EXEMPLAR_AREA_RATIO = 0.08   # ignore huge merged blobs (>8% of frame)
MIN_PATCH_SIZE = 12              # min width/height of bbox
MAX_PATCH_SIZE = 700             # max width/height of bbox
MIN_MEAN_V = 160                 # specular must be bright
MAX_MEAN_S = 140                 # specular tends to be low saturation
BBOX_PAD = 8                     # padding around bbox (captures halo)

VALID_EXT = (".png", ".jpg", ".jpeg")
# ==================================================

## 3. Detection Functions

In [ ]:
def detect_specular_mask_bgr(img_bgr):
    """HSV-based specular detection with morphological refinement."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    v = hsv[:, :, 2]
    s = hsv[:, :, 1]

    mask = ((v > HSV_V_THR) & (s < HSV_S_THR)).astype(np.uint8) * 255

    # Morphological refinement
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)

    # Dilation
    if DILATION_KERNEL_SIZE > 1:
        kd = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (DILATION_KERNEL_SIZE, DILATION_KERNEL_SIZE)
        )
        mask = cv2.dilate(mask, kd, iterations=1)

    return mask


def crop_with_pad(img, x, y, w, h, pad):
    """Crop region with padding, clamped to image bounds."""
    H, W = img.shape[:2]
    x0 = max(0, x - pad)
    y0 = max(0, y - pad)
    x1 = min(W, x + w + pad)
    y1 = min(H, y + h + pad)
    return img[y0:y1, x0:x1], (x0, y0, x1-x0, y1-y0)


def exemplar_passes_filters(img_bgr, mask_u8, bbox, frame_area):
    """Quality filter: reject non-specular patches."""
    x, y, w, h = bbox
    area = int(np.count_nonzero(mask_u8))

    if area < MIN_EXEMPLAR_AREA:                      return False, "too_small"
    if (area / frame_area) > MAX_EXEMPLAR_AREA_RATIO: return False, "too_large_ratio"
    if w < MIN_PATCH_SIZE or h < MIN_PATCH_SIZE:      return False, "bbox_too_small"
    if w > MAX_PATCH_SIZE or h > MAX_PATCH_SIZE:      return False, "bbox_too_large"

    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    m = mask_u8.astype(bool)
    if m.sum() < 10: return False, "mask_empty"

    mean_v = float(np.mean(hsv[:, :, 2][m]))
    mean_s = float(np.mean(hsv[:, :, 1][m]))

    if mean_v < MIN_MEAN_V: return False, "not_bright_enough"
    if mean_s > MAX_MEAN_S: return False, "too_saturated"

    return True, "ok"

## 4. Build the Patch Library

In [ ]:
def build_exemplar_library():
    os.makedirs(OUT_ROOT, exist_ok=True)
    patches_dir = os.path.join(OUT_ROOT, "patches")
    masks_dir   = os.path.join(OUT_ROOT, "masks")
    meta_dir    = os.path.join(OUT_ROOT, "meta")
    os.makedirs(patches_dir, exist_ok=True)
    os.makedirs(masks_dir, exist_ok=True)
    os.makedirs(meta_dir, exist_ok=True)

    files = sorted([f for f in os.listdir(FRAMES_DIR) if f.lower().endswith(VALID_EXT)])
    if not files:
        raise RuntimeError(f"No images found in {FRAMES_DIR}")

    exemplar_id = 0
    stats = {"total_frames": 0, "total_candidates": 0, "total_saved": 0, "rejected": {}}

    for fname in tqdm(files, desc="Extracting specular exemplars"):
        img_path = os.path.join(FRAMES_DIR, fname)
        img = cv2.imread(img_path)
        if img is None: continue

        stats["total_frames"] += 1
        H, W = img.shape[:2]
        frame_area = H * W

        mask = detect_specular_mask_bgr(img)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            stats["total_candidates"] += 1
            if cv2.contourArea(cnt) < MIN_CONTOUR_AREA:
                stats["rejected"]["min_contour"] = stats["rejected"].get("min_contour", 0) + 1
                continue

            x, y, w, h = cv2.boundingRect(cnt)
            patch, bbox_pad = crop_with_pad(img, x, y, w, h, BBOX_PAD)
            x0, y0, ww, hh = bbox_pad
            mask_crop = mask[y0:y0+hh, x0:x0+ww]
            mask_crop = ((mask_crop > 0).astype(np.uint8) * 255)

            ok, reason = exemplar_passes_filters(patch, mask_crop, (0, 0, ww, hh), frame_area)
            if not ok:
                stats["rejected"][reason] = stats["rejected"].get(reason, 0) + 1
                continue

            eid = f"spec_{exemplar_id:07d}"
            cv2.imwrite(os.path.join(patches_dir, f"{eid}.png"), patch)
            cv2.imwrite(os.path.join(masks_dir,   f"{eid}.png"), mask_crop)

            hsv_patch = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
            m = mask_crop.astype(bool)
            meta = {
                "id": eid,
                "source_frame": fname,
                "bbox_in_frame": [int(x0), int(y0), int(ww), int(hh)],
                "mask_area": int(np.count_nonzero(mask_crop)),
                "mean_v": float(np.mean(hsv_patch[:, :, 2][m])) if m.sum() else None,
                "mean_s": float(np.mean(hsv_patch[:, :, 1][m])) if m.sum() else None,
            }
            with open(os.path.join(meta_dir, f"{eid}.json"), "w") as f:
                json.dump(meta, f, indent=2)

            exemplar_id += 1
            stats["total_saved"] += 1

    summary_path = os.path.join(OUT_ROOT, "library_summary.json")
    with open(summary_path, "w") as f:
        json.dump(stats, f, indent=2)

    print("========================================")
    print(f"Saved exemplars : {stats['total_saved']}")
    print(f"Candidates seen : {stats['total_candidates']}")
    print(f"Frames scanned  : {stats['total_frames']}")
    print(f"Output          : {OUT_ROOT}")
    print("========================================")

build_exemplar_library()

## 5. Visualise Sample Patches

In [ ]:
import glob
import matplotlib.pyplot as plt

patches = sorted(glob.glob(f"{OUT_ROOT}/patches/*.png"))[:12]
masks   = sorted(glob.glob(f"{OUT_ROOT}/masks/*.png"))[:12]

fig, axes = plt.subplots(2, 12, figsize=(24, 4))
for i, (p, m) in enumerate(zip(patches, masks)):
    patch = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    mask  = cv2.imread(m, 0)
    axes[0, i].imshow(patch); axes[0, i].axis('off')
    axes[1, i].imshow(mask, cmap='gray'); axes[1, i].axis('off')

axes[0, 0].set_ylabel('Patch', fontsize=12)
axes[1, 0].set_ylabel('Mask',  fontsize=12)
plt.suptitle('Sample Specular Patches from Library', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/sample_patches.png", dpi=150)
plt.show()
print(f"Total patches in library: {len(glob.glob(OUT_ROOT+'/patches/*.png'))}")